In [7]:
# Импорт всех необходимых библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
import warnings

# Настройка стилей для красивых графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11

# Игнорируем предупреждения
warnings.filterwarnings('ignore')

print("✅ Библиотеки успешно импортированы!")
print(f"📌 Версия NumPy: {np.__version__}")
print(f"📌 Версия Pandas: {pd.__version__}")

✅ Библиотеки успешно импортированы!
📌 Версия NumPy: 2.4.3
📌 Версия Pandas: 3.0.1


In [9]:
# Создание демонстрационных датасетов
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler

def create_datasets(random_state=42):
    """Создает три разных набора данных для демонстрации"""
    
    # 1. Хорошо разделимые кластеры (для K-Means)
    X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, 
                                   cluster_std=0.8, random_state=random_state)
    
    # 2. Полумесяцы (для DBSCAN)
    X_moons, y_moons = make_moons(n_samples=500, noise=0.08, random_state=random_state)
    
    # 3. Концентрические окружности (для иерархической)
    X_circles, y_circles = make_circles(n_samples=500, factor=0.5, 
                                         noise=0.05, random_state=random_state)
    
    # 4. Анизотропно распределенные данные
    X_aniso = np.dot(make_blobs(n_samples=500, centers=4, random_state=random_state)[0],
                     [[0.6, -0.6], [-0.4, 0.8]])
    
    datasets = {
        'blobs': (X_blobs, y_blobs, 'Хорошо разделимые кластеры'),
        'moons': (X_moons, y_moons, 'Полумесяцы'),
        'circles': (X_circles, y_circles, 'Концентрические окружности'),
        'aniso': (X_aniso, None, 'Анизотропные данные')
    }
    
    return datasets

datasets = create_datasets()
print("📊 Датасеты созданы:")
for name, (X, y, desc) in datasets.items():
    print(f"  - {name}: {desc}, форма: {X.shape}")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Визуализация всех датасетов
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for idx, (name, (X, y, desc)) in enumerate(datasets.items()):
    axes[idx].scatter(X[:, 0], X[:, 1], s=30, alpha=0.7, 
                     edgecolors='black', linewidth=0.5)
    axes[idx].set_title(f'{name.upper()}: {desc}', fontweight='bold', fontsize=14)
    axes[idx].set_xlabel('Признак 1')
    axes[idx].set_ylabel('Признак 2')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("📈 Демонстрационные датасеты для кластеризации", 
             fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

*K-Means* — это один из самых популярных и простых алгоритмов кластеризации, который разделяет данные на K заранее заданных кластеров. Алгоритм минимизирует сумму квадратов расстояний между точками и центроидами соответствующих кластеров.

Основная идея заключается в том, чтобы найти такие центры кластеров (центроиды), чтобы суммарное расстояние от всех точек до их ближайших центроидов было минимальным.

*Алгоритм:*

* Инициализация K случайных центроидов
* Назначение каждой точки ближайшему центроиду
* Пересчет центроидов как среднее точек в кластере
* Повторение шагов 2-3 до сходимости

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA

class KMeansAnalyzer:
    """Класс для анализа и визуализации K-Means"""
    
    def __init__(self, X, name):
        self.X = X
        self.name = name
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(X)
        
    def find_optimal_k(self, max_k=10):
        """Находит оптимальное K используя метод локтя и силуэт"""
        
        inertias = []
        silhouettes = []
        calinski_scores = []
        K_range = range(2, max_k + 1)
        
        for k in K_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = kmeans.fit_predict(self.X_scaled)
            
            inertias.append(kmeans.inertia_)
            silhouettes.append(silhouette_score(self.X_scaled, labels))
            calinski_scores.append(calinski_harabasz_score(self.X_scaled, labels))
        
        # Визуализация
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Метод локтя
        axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
        axes[0].set_xlabel('K (количество кластеров)')
        axes[0].set_ylabel('Инерция')
        axes[0].set_title('Метод локтя\n(ищите точку изгиба)', fontweight='bold')
        axes[0].grid(True, alpha=0.3)
        
        # Силуэт
        axes[1].plot(K_range, silhouettes, 'ro-', linewidth=2, markersize=8)
        axes[1].set_xlabel('K (количество кластеров)')
        axes[1].set_ylabel('Коэффициент силуэта')
        axes[1].set_title('Коэффициент силуэта\n(чем выше, тем лучше)', fontweight='bold')
        axes[1].grid(True, alpha=0.3)
        
        # Calinski-Harabasz
        axes[2].plot(K_range, calinski_scores, 'go-', linewidth=2, markersize=8)
        axes[2].set_xlabel('K (количество кластеров)')
        axes[2].set_ylabel('Индекс Calinski-Harabasz')
        axes[2].set_title('Индекс Calinski-Harabasz\n(чем выше, тем лучше)', fontweight='bold')
        axes[2].grid(True, alpha=0.3)
        
        plt.suptitle(f"🔍 Выбор оптимального K для {self.name}", 
                    fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Рекомендация
        optimal_k = np.argmax(silhouettes) + 2
        print(f"📊 Рекомендуемое K: {optimal_k} (на основе максимального силуэта)")
        return optimal_k
    
    def apply_kmeans(self, n_clusters):
        """Применяет K-Means и возвращает результаты"""
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = kmeans.fit_predict(self.X_scaled)
        
        # Метрики
        metrics = {
            'silhouette': silhouette_score(self.X_scaled, labels),
            'calinski': calinski_harabasz_score(self.X_scaled, labels),
            'davies': davies_bouldin_score(self.X_scaled, labels),
            'inertia': kmeans.inertia_
        }
        
        return labels, kmeans.cluster_centers_, metrics
    
    def visualize_results(self, labels, centers):
        """Визуализирует результаты кластеризации"""
        
        # Возвращаем центроиды в исходный масштаб
        centers_original = self.scaler.inverse_transform(centers)
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Исходные данные
        axes[0].scatter(self.X[:, 0], self.X[:, 1], s=30, alpha=0.6,
                       edgecolors='black', linewidth=0.5)
        axes[0].set_title(f'{self.name}: Исходные данные', fontweight='bold')
        axes[0].set_xlabel('Признак 1')
        axes[0].set_ylabel('Признак 2')
        axes[0].grid(True, alpha=0.3)
        
        # Результаты кластеризации
        scatter = axes[1].scatter(self.X[:, 0], self.X[:, 1], c=labels, s=30, 
                                  alpha=0.7, cmap='viridis', 
                                  edgecolors='black', linewidth=0.5)
        axes[1].scatter(centers_original[:, 0], centers_original[:, 1], 
                       marker='X', s=200, c='red', edgecolors='black', 
                       linewidth=2, label='Центроиды')
        axes[1].set_title(f'{self.name}: K-Means кластеризация', fontweight='bold')
        axes[1].set_xlabel('Признак 1')
        axes[1].set_ylabel('Признак 2')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.colorbar(scatter, ax=axes[1], label='Кластер')
        plt.tight_layout()
        plt.show()

# Применяем K-Means к блобам
print("🔬 Анализ K-Means на данных 'blobs'")
print("=" * 50)

kmeans_analyzer = KMeansAnalyzer(datasets['blobs'][0], "Blobs")
optimal_k = kmeans_analyzer.find_optimal_k(max_k=8)
labels, centers, metrics = kmeans_analyzer.apply_kmeans(optimal_k)
kmeans_analyzer.visualize_results(labels, centers)

print("\n📊 Метрики качества:")
print(f"  - Silhouette Score: {metrics['silhouette']:.3f} (чем ближе к 1, тем лучше)")
print(f"  - Calinski-Harabasz: {metrics['calinski']:.1f} (чем больше, тем лучше)")
print(f"  - Davies-Bouldin: {metrics['davies']:.3f} (чем меньше, тем лучше)")
print(f"  - Инерция: {metrics['inertia']:.1f}")

*Иерархическая кластеризация* строит дерево (дендрограмму) кластеров, объединяя или разделяя группы на разных уровнях.

*Методы связи (linkage):*

- Ward: минимизирует дисперсию внутри кластеров
- Complete: максимальное расстояние между точками
- Average: среднее расстояние между точками
- Single: минимальное расстояние между точками

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

class HierarchicalAnalyzer:
    """Класс для анализа и визуализации иерархической кластеризации"""
    
    def __init__(self, X, name):
        self.X = X
        self.name = name
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(X)
    
    def plot_dendrogram(self, method='ward', truncate_level=5):
        """Рисует дендрограмму"""
        
        # Вычисляем матрицу связей
        linkage_matrix = linkage(self.X_scaled, method=method)
        
        plt.figure(figsize=(14, 7))
        dendrogram(linkage_matrix, truncate_mode='level', p=truncate_level,
                  color_threshold=0, above_threshold_color='gray')
        plt.title(f'Дендрограмма для {self.name}\nМетод: {method}', 
                 fontsize=16, fontweight='bold')
        plt.xlabel('Индексы образцов')
        plt.ylabel('Расстояние')
        plt.grid(True, alpha=0.3)
        plt.axhline(y=np.median(linkage_matrix[:, 2]), color='red', 
                   linestyle='--', alpha=0.7, label='Порог отсечения')
        plt.legend()
        plt.tight_layout()
        plt.show()
        
        return linkage_matrix
    
    def compare_linkage_methods(self, n_clusters):
        """Сравнивает различные методы связи"""
        
        methods = ['ward', 'complete', 'average', 'single']
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.ravel()
        
        results = {}
        
        for idx, method in enumerate(methods):
            hierarchical = AgglomerativeClustering(n_clusters=n_clusters, 
                                                  linkage=method)
            labels = hierarchical.fit_predict(self.X_scaled)
            
            # Сохраняем результаты
            results[method] = labels
            
            # Визуализация
            scatter = axes[idx].scatter(self.X[:, 0], self.X[:, 1], 
                                       c=labels, s=30, alpha=0.7,
                                       cmap='viridis', edgecolors='black', linewidth=0.5)
            axes[idx].set_title(f'Метод: {method}', fontweight='bold')
            axes[idx].set_xlabel('Признак 1')
            axes[idx].set_ylabel('Признак 2')
            axes[idx].grid(True, alpha=0.3)
            
            # Добавляем метрику, если есть >1 кластера
            if len(set(labels)) > 1:
                sil = silhouette_score(self.X_scaled, labels)
                axes[idx].text(0.05, 0.95, f'Silhouette: {sil:.3f}', 
                              transform=axes[idx].transAxes, fontsize=10,
                              bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.suptitle(f"🔍 Сравнение методов связи для {self.name}", 
                    fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        return results
    
    def apply_hierarchical(self, n_clusters, method='ward'):
        """Применяет иерархическую кластеризацию"""
        
        hierarchical = AgglomerativeClustering(n_clusters=n_clusters, 
                                              linkage=method)
        labels = hierarchical.fit_predict(self.X_scaled)
        
        # Метрики
        metrics = {
            'silhouette': silhouette_score(self.X_scaled, labels)
        }
        
        return labels, metrics
    
    def visualize_results(self, labels, n_clusters, method):
        """Визуализирует результаты"""
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Исходные данные
        axes[0].scatter(self.X[:, 0], self.X[:, 1], s=30, alpha=0.6,
                       edgecolors='black', linewidth=0.5)
        axes[0].set_title(f'{self.name}: Исходные данные', fontweight='bold')
        axes[0].set_xlabel('Признак 1')
        axes[0].set_ylabel('Признак 2')
        axes[0].grid(True, alpha=0.3)
        
        # Результаты
        scatter = axes[1].scatter(self.X[:, 0], self.X[:, 1], c=labels, s=30, 
                                  alpha=0.7, cmap='viridis', 
                                  edgecolors='black', linewidth=0.5)
        axes[1].set_title(f'{self.name}: Иерархическая кластеризация\n'
                         f'Метод: {method}, K={n_clusters}', fontweight='bold')
        axes[1].set_xlabel('Признак 1')
        axes[1].set_ylabel('Признак 2')
        axes[1].grid(True, alpha=0.3)
        
        plt.colorbar(scatter, ax=axes[1], label='Кластер')
        plt.tight_layout()
        plt.show()

# Применяем иерархическую кластеризацию к окружностям
print("🔬 Анализ иерархической кластеризации на данных 'circles'")
print("=" * 50)

hier_analyzer = HierarchicalAnalyzer(datasets['circles'][0], "Circles")

# Рисуем дендрограмму
linkage_matrix = hier_analyzer.plot_dendrogram(method='ward', truncate_level=5)

# Сравниваем методы
results = hier_analyzer.compare_linkage_methods(n_clusters=2)

# Применяем лучший метод
labels, metrics = hier_analyzer.apply_hierarchical(n_clusters=2, method='ward')
hier_analyzer.visualize_results(labels, n_clusters=2, method='ward')

print(f"\n📊 Метрики качества:")
print(f"  - Silhouette Score: {metrics['silhouette']:.3f}")